In [8]:
import polars as pl
from scipy.stats import norm, poisson
import numpy as np
import datetime
import tensorflow as tf
import plotly.graph_objects as go
from typing import Literal
from src.dataLoader import getResultDataframe

In [9]:
dfResult = getResultDataframe()
if dfResult is None:
    print("No data found. Please make sure the results.csv file is in the data/ directory.")
    dfResult = pl.DataFrame()

In [10]:
dfResult

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
str,str,str,i64,i64,str,str,str,bool
"""1872-11-30""","""Scotland""","""England""",0,0,"""Friendly""","""Glasgow""","""Scotland""",false
"""1873-03-08""","""England""","""Scotland""",4,2,"""Friendly""","""London""","""England""",false
"""1874-03-07""","""Scotland""","""England""",2,1,"""Friendly""","""Glasgow""","""Scotland""",false
"""1875-03-06""","""England""","""Scotland""",2,2,"""Friendly""","""London""","""England""",false
"""1876-03-04""","""Scotland""","""England""",3,0,"""Friendly""","""Glasgow""","""Scotland""",false
…,…,…,…,…,…,…,…,…
"""2026-06-27""","""Jordan""","""Argentina""",null,null,"""FIFA World Cup""","""Arlington""","""United States""",true
"""2026-06-27""","""Colombia""","""Portugal""",null,null,"""FIFA World Cup""","""Miami Gardens""","""United States""",true
"""2026-06-27""","""DR Congo""","""Uzbekistan""",null,null,"""FIFA World Cup""","""Atlanta""","""United States""",true


In [12]:
allTeams = dfResult["home_team"].unique().to_list() + dfResult["away_team"].unique().to_list()
allTeams = np.sort(list(set(allTeams)))
defendingParams = np.random.normal(1.5,0.4, len(allTeams))
attackingParams = np.random.normal(1.5,0.4, len(allTeams))
dfParams = pl.DataFrame({
    "team": allTeams,
    "num": range(len(allTeams)),
    "defending": defendingParams,
    "attacking": attackingParams
})
dictNum = {}
for i, team in enumerate(allTeams):
    dictNum[str(team)] = i
numTeams = len(allTeams)



In [13]:
def getMatchMatrice(
    dfMatch: pl.DataFrame, 
    scope: pl.Expr,
    teamNumerotation: dict[str, int]
    ) -> np.ndarray:
    numTeams = len(teamNumerotation)
    df = dfMatch.filter(scope)
    res = np.zeros((df.height*2, numTeams*2 + 3))
    for i, row in enumerate(df.iter_rows(named=True)):
        #home team score
        res[2*i, teamNumerotation[str(row["home_team"])]] = 1
        res[2*i, numTeams+teamNumerotation[str(row["away_team"])]] = 1
        res[2*i, -1] = row["home_score"]
        #away team score
        res[2*i + 1, numTeams+teamNumerotation[str(row["home_team"])]] = 1
        res[2*i + 1, teamNumerotation[str(row["away_team"])]] = 1
        res[2*i + 1, -1] = row["away_score"]

        if not row["neutral"]:
            res[2*i, -3] = 1
            res[2*i + 1, -2] = 1
    return res

In [16]:
scope = pl.lit(True)
scope = scope & (pl.col("date").is_between(datetime.date(2025, 6, 1), datetime.date(2026, 6, 1)))
scope = scope & (pl.all_horizontal([pl.col(column).is_not_null() for column in dfResult.columns]))

matchMatrice = getMatchMatrice(dfResult, scope, dictNum)
matchMatrice


array([[0., 0., 0., ..., 1., 0., 2.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 1., 0., ..., 0., 1., 1.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 3.]], shape=(1976, 675))

In [17]:
class MyModel(tf.keras.Model):
    def __init__(
        self,
        numTeams: int
    ):
        super().__init__()
        self.numTeams = numTeams
        self.teamParams = self.add_weight(
            name="team_params",
            shape=(2 * numTeams, 1),
            initializer=tf.random_normal_initializer(),
            trainable=True
        )
        self.awayHomeMultiplier = self.add_weight(
            name="away_home_multiplier",
            shape=(1, 2),
            initializer=tf.random_normal_initializer(),
            trainable=True
        )


    def call(self, input):
        attackingParams = self.teamParams[:self.numTeams]
        defendingParams = self.teamParams[self.numTeams:2*self.numTeams]

        attackingOneHotEncoder = input[:,:self.numTeams]
        defendingOneHotEncoder = input[:,self.numTeams:2*self.numTeams]
        score = input[:,-1]
        awayHomeMultiplier = tf.reduce_sum(input[:, -3:-1] * self.awayHomeMultiplier, axis=1, keepdims=True)
        scoreExpanded = tf.expand_dims(score, axis=1)

        params = tf.exp(awayHomeMultiplier + tf.matmul(attackingOneHotEncoder, attackingParams) - tf.matmul(defendingOneHotEncoder, defendingParams))
        
        poissonLogLikelyHood = -params + tf.math.log(params) * scoreExpanded - tf.math.lgamma(scoreExpanded + 1)

        return tf.reduce_sum(poissonLogLikelyHood)

In [18]:
X = tf.constant(matchMatrice, dtype=tf.float32)
model = MyModel(numTeams=numTeams)

In [19]:

model = MyModel(numTeams=numTeams)
optimizer = tf.keras.optimizers.Adam()
X = tf.constant(matchMatrice, dtype=tf.float32)
numEpoch = 10000
lossHistory = []
for i in range(numEpoch):
    print(f"Epoch {i+1}/{numEpoch}", end="\r")
    with tf.GradientTape() as tape:
        pred = -model(X)
        loss = pred
    lossHistory.append(loss)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

In [20]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=lossHistory, mode='lines', name='Loss'))
fig.update_layout(title='Loss History', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

In [21]:
attack = [np.exp(model.teamParams.numpy())[i][0] for i in range(numTeams)]
defend = [np.exp(model.teamParams.numpy())[numTeams+i][0] for i in range(numTeams)]

In [22]:
dictAttack = {str(allTeams[i]): attack[i] for i in range(numTeams)}
dictDefend = {str(allTeams[i]): defend[i] for i in range(numTeams)}
dictAwayHome = {"home": np.exp(model.awayHomeMultiplier.numpy())[0][0], "away": np.exp(model.awayHomeMultiplier.numpy())[0][1]}

In [23]:

def getStatsDf(
    teamA: str, 
    teamB: str, 
    maxScore: int, 
    teamHome : Literal["first", "second", "neutral"]
    ) -> pl.DataFrame:
    dfDict = {
        teamA: [],
        teamB: [],
        "probability": []
    }
    multiplierA, multiplierB = 1, 1
    if teamHome == "first":
        multiplierA = dictAwayHome["home"]
        multiplierB = dictAwayHome["away"]
    elif teamHome == "second":
        multiplierA = dictAwayHome["away"]
        multiplierB = dictAwayHome["home"]

    for i in range(maxScore + 1):
        for j in range(maxScore + 1):
            dfDict[teamA].append(i)
            dfDict[teamB].append(j)
            prob = 100*poisson.pmf(i, multiplierA * dictAttack[teamA] / dictDefend[teamB]) * poisson.pmf(j, multiplierB * dictAttack[teamB] / dictDefend[teamA])
            dfDict["probability"].append(prob)
    res= pl.DataFrame(dfDict).sort("probability", descending=True)
    teamAWins = pl.col(teamA) > pl.col(teamB)
    teamBWins = pl.col(teamB) > pl.col(teamA)
    res = res.with_columns(
        result=pl.when(teamAWins).then(pl.lit(teamA)).when(teamBWins).then(pl.lit(teamB)).otherwise(pl.lit("draw")).alias("result")
    )
    resSummedUp = res.group_by("result").agg(pl.col("probability").sum())
    res = res.join(resSummedUp, on="result", how="left").rename({"probability_right": "result_probability"})
    res = res.with_columns(
        expectedGain = (3*pl.col("probability") + pl.col("result_probability") - pl.col("probability"))/100)
    return res

In [38]:
teamA = "Mexico"
teamB = "South Africa"
probScore = getStatsDf(teamA, teamB, 10, "first")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Mexico,South Africa,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
1,0,21.128443,"""Mexico""",68.065755,1.103226
2,0,17.308228,"""Mexico""",68.065755,1.026822
3,0,9.452495,"""Mexico""",68.065755,0.869707
2,1,7.09425,"""Mexico""",68.065755,0.822543
3,1,3.874363,"""Mexico""",68.065755,0.758145
…,…,…,…,…,…
8,9,1.4941e-11,"""South Africa""",8.811079,0.088111
6,10,1.2776e-11,"""South Africa""",8.811079,0.088111
7,10,2.9902e-12,"""South Africa""",8.811079,0.088111


In [39]:
teamA = "South Korea"
teamB = "Czech Republic"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

South Korea,Czech Republic,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
2,1,9.789861,"""South Korea""",65.949242,0.85529
2,0,9.77232,"""South Korea""",65.949242,0.854939
1,0,8.602547,"""South Korea""",65.949242,0.831543
3,1,7.414056,"""South Korea""",65.949242,0.807774
3,0,7.400773,"""South Korea""",65.949242,0.807508
…,…,…,…,…,…
6,10,2.0292e-7,"""Czech Republic""",15.306362,0.153064
8,9,1.8671e-7,"""Czech Republic""",15.306362,0.153064
7,10,6.5861e-8,"""Czech Republic""",15.306362,0.153064


In [41]:
teamA = "Canada"
teamB = "Bosnia and Herzegovina"
probScore = getStatsDf(teamA, teamB, 10, "first")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Canada,Bosnia and Herzegovina,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
1,0,21.5127,"""Canada""",69.840171,1.128656
2,0,18.015692,"""Canada""",69.840171,1.058716
3,0,10.058095,"""Canada""",69.840171,0.899564
2,1,6.798845,"""Canada""",69.840171,0.834379
4,0,4.211548,"""Canada""",69.840171,0.782633
…,…,…,…,…,…
8,9,8.4406e-12,"""Bosnia and Herzegovina""",7.820214,0.078202
6,10,6.3587e-12,"""Bosnia and Herzegovina""",7.820214,0.078202
7,10,1.5215e-12,"""Bosnia and Herzegovina""",7.820214,0.078202


In [47]:
teamA = "United States"
teamB = "Paraguay"
probScore = getStatsDf(teamA, teamB, 10, "first")
probScore = probScore.sort("expectedGain", descending=True)
probScore

United States,Paraguay,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
0,1,9.810563,"""Paraguay""",50.592211,0.702133
1,2,9.63601,"""Paraguay""",50.592211,0.698642
0,2,8.399115,"""Paraguay""",50.592211,0.673904
1,3,5.499784,"""Paraguay""",50.592211,0.615918
0,3,4.793822,"""Paraguay""",50.592211,0.601799
…,…,…,…,…,…
6,6,0.000635,"""draw""",23.879353,0.238806
7,7,0.000025,"""draw""",23.879353,0.238794
8,8,7.8154e-7,"""draw""",23.879353,0.238794


In [43]:
teamA = "Switzerland"
teamB = "Qatar"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Switzerland,Qatar,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
7,0,10.226488,"""Switzerland""",89.051183,1.095042
6,0,10.092132,"""Switzerland""",89.051183,1.092354
8,0,9.067305,"""Switzerland""",89.051183,1.071858
5,0,8.536749,"""Switzerland""",89.051183,1.061247
9,0,7.146236,"""Switzerland""",89.051183,1.033437
…,…,…,…,…,…
3,10,5.2512e-11,"""Qatar""",0.076003,0.00076
0,9,2.3493e-11,"""Qatar""",0.076003,0.00076
2,10,2.2209e-11,"""Qatar""",0.076003,0.00076


In [44]:
teamA = "Brazil"
teamB = "Morocco"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Brazil,Morocco,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
0,0,18.002418,"""draw""",33.849288,0.698541
1,0,15.994216,"""Brazil""",34.808766,0.667972
0,1,14.873884,"""Morocco""",31.341945,0.610897
1,1,13.214676,"""draw""",33.849288,0.602786
2,0,7.105016,"""Brazil""",34.808766,0.490188
…,…,…,…,…,…
6,10,5.0230e-10,"""Morocco""",31.341945,0.313419
8,9,8.5693e-11,"""Morocco""",31.341945,0.313419
7,10,6.3753e-11,"""Morocco""",31.341945,0.313419


In [45]:
teamA = "Haiti"
teamB = "Scotland"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Haiti,Scotland,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
0,2,12.062313,"""Scotland""",73.494158,0.976188
0,1,10.040976,"""Scotland""",73.494158,0.935761
0,3,9.660376,"""Scotland""",73.494158,0.928149
1,2,9.317377,"""Scotland""",73.494158,0.921289
1,3,7.462032,"""Scotland""",73.494158,0.884182
…,…,…,…,…,…
9,8,3.1049e-8,"""Haiti""",10.132898,0.101329
10,6,2.3267e-8,"""Haiti""",10.132898,0.101329
10,7,7.9859e-9,"""Haiti""",10.132898,0.101329


In [46]:
teamA = "Australia"
teamB = "Turkey"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Australia,Turkey,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
2,1,8.793747,"""Australia""",40.999542,0.58587
1,0,8.161701,"""Australia""",40.999542,0.573229
2,0,6.29754,"""Australia""",40.999542,0.535946
1,2,7.957137,"""Turkey""",34.488844,0.504031
3,1,4.523483,"""Australia""",40.999542,0.500465
…,…,…,…,…,…
6,6,0.001021,"""draw""",24.511514,0.245136
7,7,0.000045,"""draw""",24.511514,0.245116
8,8,0.000002,"""draw""",24.511514,0.245115


In [51]:
teamA = "Netherlands"
teamB = "Japan"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

Netherlands,Japan,probability,result,result_probability,expectedGain
i64,i64,f64,str,f64,f64
1,0,10.533545,"""Netherlands""",37.399001,0.584661
0,1,10.200548,"""Japan""",35.49291,0.55894
2,1,8.114149,"""Netherlands""",37.399001,0.536273
1,1,12.866139,"""draw""",27.108072,0.528404
1,2,7.857637,"""Japan""",35.49291,0.512082
…,…,…,…,…,…
6,6,0.000215,"""draw""",27.108072,0.271085
7,7,0.000007,"""draw""",27.108072,0.271081
8,8,1.6304e-7,"""draw""",27.108072,0.271081


In [29]:
teamAWins = probScore.filter(pl.col(teamA) > pl.col(teamB)).select(pl.col("probability").sum())[0,0]
teamBWins = probScore.filter(pl.col(teamB) > pl.col(teamA)).select(pl.col("probability").sum())[0,0]
draws = probScore.filter(pl.col(teamA) == pl.col(teamB)).select(pl.col("probability").sum())[0,0]
print(f"{teamA} wins: {teamAWins:.2f}%")
print(f"{teamB} wins: {teamBWins:.2f}%")
print(f"Draw: {draws:.2f}%")

Switzerland wins: 89.05%
Qatar wins: 0.08%
Draw: 0.35%


In [30]:
teamBWins

0.07600326636251822

In [31]:
dictAttack["South Africa"]

np.float32(1.7621456)

In [32]:
dictDefend["Mexico"]

np.float32(3.2489254)

In [33]:
dictDefend["South Africa"]

np.float32(1.4767506)

In [34]:
df = pl.DataFrame({
    "team": allTeams,
    "attack": attack,
    "defend": defend
})

In [35]:
df

team,attack,defend
str,f32,f32
"""Abkhazia""",1.044979,0.923829
"""Afghanistan""",0.279899,1.882492
"""Albania""",1.202819,1.914672
"""Alderney""",0.92874,0.987386
"""Algeria""",2.860874,2.432289
…,…,…
"""Yugoslavia""",1.025531,0.998298
"""Zambia""",1.070424,0.930199
"""Zanzibar""",1.058106,0.979473


In [36]:
defend

[np.float32(0.92382944),
 np.float32(1.8824924),
 np.float32(1.9146717),
 np.float32(0.98738605),
 np.float32(2.4322894),
 np.float32(1.0553608),
 np.float32(0.014829345),
 np.float32(1.0611525),
 np.float32(0.86217606),
 np.float32(1.0563145),
 np.float32(0.14371249),
 np.float32(0.4014761),
 np.float32(1.0604836),
 np.float32(5.865772),
 np.float32(0.77195626),
 np.float32(1.0829071),
 np.float32(0.36109927),
 np.float32(1.0166723),
 np.float32(2.6651092),
 np.float32(3.339317),
 np.float32(0.9839398),
 np.float32(1.2733492),
 np.float32(0.13357161),
 np.float32(0.8122601),
 np.float32(0.31903607),
 np.float32(0.9522815),
 np.float32(0.24384576),
 np.float32(137.15512),
 np.float32(1.2105213),
 np.float32(1.44918),
 np.float32(0.43527353),
 np.float32(1.4818473),
 np.float32(0.35042086),
 np.float32(0.20936248),
 np.float32(1.0475881),
 np.float32(1.8704505),
 np.float32(0.28080064),
 np.float32(1.6307778),
 np.float32(0.8832452),
 np.float32(3.1684725),
 np.float32(0.2835322),
 np.f

In [37]:
attack

[np.float32(1.0449789),
 np.float32(0.27989933),
 np.float32(1.2028192),
 np.float32(0.92873955),
 np.float32(2.8608744),
 np.float32(1.0118822),
 np.float32(0.20193313),
 np.float32(0.940445),
 np.float32(0.36129552),
 np.float32(1.5893075),
 np.float32(0.19574328),
 np.float32(0.12893632),
 np.float32(1.0634589),
 np.float32(3.2371392),
 np.float32(1.6274439),
 np.float32(1.0104213),
 np.float32(0.64610183),
 np.float32(0.9900791),
 np.float32(2.8332238),
 np.float32(2.7885067),
 np.float32(0.9573675),
 np.float32(1.2002716),
 np.float32(0.08911645),
 np.float32(1.1453863),
 np.float32(0.53330386),
 np.float32(1.0263541),
 np.float32(0.47848204),
 np.float32(6.070295),
 np.float32(1.9184173),
 np.float32(3.9990456),
 np.float32(0.93076366),
 np.float32(1.279311),
 np.float32(1.0205803),
 np.float32(0.20794706),
 np.float32(1.0453966),
 np.float32(1.4926038),
 np.float32(0.58408034),
 np.float32(2.1757438),
 np.float32(0.6339061),
 np.float32(5.2354565),
 np.float32(0.588946),
 np.flo